# ALDIMI - Predicciones

Modelo base para predecir Consumo_Diario con el dataset merged.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
try:
    from xgboost import XGBRegressor
except ModuleNotFoundError:
    from sklearn.ensemble import GradientBoostingRegressor as XGBRegressor
    print("xgboost is not installed; using GradientBoostingRegressor as fallback.")
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
BASE_DIR = Path.cwd()
if BASE_DIR.name == 'src':
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / 'data' / 'raw' / 'stock_raw.csv'
df = pd.read_csv(DATA_PATH)
print('Loaded:', df.shape)

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])

agg_map = {
    'Inventory_Level': 'mean'
}
if 'Units_Sold' in df.columns:
    agg_map['Units_Sold'] = 'sum'
if 'Supplier_Lead_Time_Days' in df.columns:
    agg_map['Supplier_Lead_Time_Days'] = 'mean'
if 'Reorder_Point' in df.columns:
    agg_map['Reorder_Point'] = 'mean'
if 'Order_Quantity' in df.columns:
    agg_map['Order_Quantity'] = 'sum'
if 'Demand_Forecast' in df.columns:
    agg_map['Demand_Forecast'] = 'mean'

if 'Inventory_Level' not in df.columns:
    raise ValueError('Inventory_Level not found in dataset')

daily = df.groupby('Date').agg(agg_map).sort_index()

# Feature Engineering: Lags and Rolling Windows
for lag in [1, 7, 14]:
    daily[f'Inventory_Level_lag_{lag}'] = daily['Inventory_Level'].shift(lag)
    if 'Units_Sold' in daily.columns:
        daily[f'Units_Sold_lag_{lag}'] = daily['Units_Sold'].shift(lag)

daily['Inventory_Level_roll7'] = daily['Inventory_Level'].rolling(7).mean()
if 'Units_Sold' in daily.columns:
    daily['Units_Sold_roll7'] = daily['Units_Sold'].rolling(7).mean()

# Target variables for horizons
daily['Inventory_Level_t+7'] = daily['Inventory_Level'].shift(-7)
daily['Inventory_Level_t+14'] = daily['Inventory_Level'].shift(-14)

data = daily.dropna().copy()
print('Prepared time series rows:', len(data))

feature_cols = [c for c in data.columns if not c.startswith('Inventory_Level_t+')]

# Split Train/Test (Temporal Split)
split_idx = int(len(data) * 0.8)
train = data.iloc[:split_idx]
test = data.iloc[split_idx:]

X_train = train[feature_cols]
X_test = test[feature_cols]

def train_and_evaluate(horizon):
    target_col = f'Inventory_Level_t+{horizon}'
    y_train = train[target_col]
    y_test = test[target_col]
    
    print(f'\n=== Horizon t+{horizon} days ===')
    
    # 1. Random Forest Tuning
    rf_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10],
        'min_samples_split': [2, 5]
    }
    rf_search = GridSearchCV(RandomForestRegressor(random_state=42), rf_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    rf_search.fit(X_train, y_train)
    best_rf = rf_search.best_estimator_
    
    # 2. XGBoost Tuning
    xgb_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    }
    xgb_search = GridSearchCV(XGBRegressor(random_state=42), xgb_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    xgb_search.fit(X_train, y_train)
    best_xgb = xgb_search.best_estimator_
    
    # Predictions
    rf_preds = best_rf.predict(X_test)
    xgb_preds = best_xgb.predict(X_test)
    
    # Metrics
    for name, preds in [('RandomForest', rf_preds), ('XGBoost', xgb_preds)]:
        mae = mean_absolute_error(y_test, preds)
        rmse = mean_squared_error(y_test, preds)**0.5
        r2 = r2_score(y_test, preds)
        print(f'{name} - MAE: {mae:.3f} | RMSE: {rmse:.3f} | R2: {r2:.3f}')
    
    # Plotting
    plt.figure(figsize=(15, 6))
    plt.plot(test.index, y_test, label='Actual', color='black', linewidth=2, alpha=0.7)
    plt.plot(test.index, rf_preds, label='Random Forest Pred', linestyle='--', alpha=0.8)
    plt.plot(test.index, xgb_preds, label='XGBoost Pred', linestyle=':', alpha=0.8)
    
    plt.title(f'Gráfico 1: Predicción de Niveles de Inventario (Horizonte t+{horizon})', fontsize=16, fontweight='bold')
    plt.xlabel('Fecha', fontsize=12)
    plt.ylabel('Nivel de Inventario', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    description = f"Descripción: Comparativa entre los valores reales y las predicciones de los modelos para un horizonte de {horizon} días. El modelo busca anticipar cambios en el stock para evitar desabastecimiento."
    plt.figtext(0.5, -0.05, description, ha="center", fontsize=10, style='italic')
    
    plt.tight_layout()
    plt.show()

# Run for both horizons
for h in [7, 14]:
    train_and_evaluate(h)

Loaded: (91250, 15)
Prepared time series rows: 337

=== Horizon t+7 days ===


TypeError: got an unexpected keyword argument 'squared'